# Daily ETF history: QLD, QQQ, UBT, TLT, USD, ROM

Run **all cells in order** in a Python 3.10+ Jupyter kernel. The first cell installs dependencies into that kernel. Internet access is required. Outputs go to `data/` relative to the kernel working directory.

Downloads every available daily observation from Yahoo Finance via yfinance, ending **before today in New York** to avoid partial daily bars. The main CSV is the longest common observed history across all six ETFs; no prices are filled or synthetic pre-inception data created. Source gaps are reported and incomplete joint rows excluded. A second CSV preserves the full union of histories, including dates before all funds existed.

|Ticker|Fund|
|---|---|
|QLD|ProShares Ultra QQQ|
|QQQ|Invesco QQQ Trust|
|UBT|ProShares Ultra 20+ Year Treasury|
|TLT|iShares 20+ Year Treasury Bond ETF|
|USD|ProShares Ultra Semiconductors (an ETF, not a currency series)|
|ROM|ProShares Ultra Technology|

Sources: [yfinance documentation](https://ranaroussi.github.io/yfinance/), [USD](https://www.proshares.com/our-etfs/leveraged-and-inverse/usd), [ROM](https://www.proshares.com/our-etfs/leveraged-and-inverse/rom).

Yahoo historical data can be revised. Adjusted prices incorporate subsequent corporate actions and are not point-in-time snapshots. These are ETF prices, not bond yield series. Yahoo data access is intended for personal research; consult the source terms before redistribution.


In [ ]:
%pip install -q "yfinance>=0.2.54,<2" "pandas>=2.2,<4" "numpy>=1.26,<3"

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib.metadata import version
import json
import time
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import display

TICKERS = ["QLD", "QQQ", "UBT", "TLT", "USD", "ROM"]
START = "1900-01-01"  # Request all available history, not a guessed inception date.
END_EXCLUSIVE = datetime.now(ZoneInfo("America/New_York")).date().isoformat()
OUTPUT_DIR = Path.cwd() / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUTPUT_DIR.resolve()}")
print(f"Request: {START} <= date < {END_EXCLUSIVE}")


## Download and validate each fund

`auto_adjust=False` retains Yahoo's OHLC and separate adjusted close. Yahoo OHLC is generally split adjusted; “close” should not be interpreted as an original historical tape price. `adj_close` also adjusts for distributions. `return_adj` is its one-observation percentage change, expressed as a decimal. It approximates a reinvested-distribution return and is calculated on each fund's own history **before** alignment. `return_gap_days` gives the calendar-day distance to the preceding observation, making gaps visible. Returns are missing on the first observation.

Corporate-action columns are cash distributions per share and split ratios (zero means no event). Volume is the provider's reported share volume. No silent price repair is applied. Retries handle transient requests; any missing ticker or invalid price prevents a final export.


In [ ]:
PRICE_FIELDS = ["Open", "High", "Low", "Close", "Adj Close"]
RENAME = {"Open": "open", "High": "high", "Low": "low", "Close": "close",
          "Adj Close": "adj_close", "Volume": "volume", "Dividends": "dividends",
          "Stock Splits": "stock_splits", "Capital Gains": "capital_gains"}
REQUIRED = ["open", "high", "low", "close", "adj_close", "volume"]

def clean_history(raw, ticker):
    if raw is None or raw.empty:
        raise ValueError(f"{ticker}: provider returned no history")
    missing = set(PRICE_FIELDS + ["Volume"]) - set(raw.columns)
    if missing:
        raise ValueError(f"{ticker}: missing required columns {sorted(missing)}")
    frame = raw.copy()
    dates = pd.DatetimeIndex(frame.index)
    # Preserve exchange-local dates; do not convert midnight bars to UTC dates.
    if dates.tz is not None:
        dates = dates.tz_localize(None)
    frame.index = dates.normalize()
    frame.index.name = "date"
    frame = frame.sort_index()
    if frame.index.has_duplicates:
        raise ValueError(f"{ticker}: duplicate trading dates")
    frame = frame.loc[(frame.index >= pd.Timestamp(START)) &
                      (frame.index < pd.Timestamp(END_EXCLUSIVE))]
    # Action-only rows have no price bar; preserve real bars and report removals.
    no_price = frame[PRICE_FIELDS].isna().all(axis=1)
    print(f"{ticker}: excluding {int(no_price.sum())} rows without price bars")
    frame = frame.loc[~no_price].copy()
    if frame.empty:
        raise ValueError(f"{ticker}: no price bars within requested range")
    for field in ["Dividends", "Stock Splits", "Capital Gains"]:
        if field not in frame:
            frame[field] = 0.0
    frame = frame[list(RENAME)].rename(columns=RENAME).apply(pd.to_numeric, errors="raise")
    values = frame[REQUIRED].to_numpy(dtype=float)
    if np.isinf(values).any():
        raise ValueError(f"{ticker}: infinite price or volume")
    if (frame[REQUIRED[:-1]] <= 0).any().any() or (frame.volume < 0).any():
        raise ValueError(f"{ticker}: nonpositive price or negative volume")
    if ((frame.high < frame[["open", "close", "low"]].max(axis=1) - 1e-6) |
        (frame.low > frame[["open", "close", "high"]].min(axis=1) + 1e-6)).any():
        raise ValueError(f"{ticker}: inconsistent OHLC bars")
    frame["return_adj"] = frame.adj_close.pct_change(fill_method=None)
    frame["return_gap_days"] = frame.index.to_series().diff().dt.days
    return frame

histories = {}
for ticker in TICKERS:
    for attempt in range(1, 4):
        try:
            raw = yf.Ticker(ticker).history(
                start=START, end=END_EXCLUSIVE, interval="1d",
                auto_adjust=False, back_adjust=False, actions=True,
                repair=False, keepna=True, raise_errors=True, timeout=30,
            )
            histories[ticker] = clean_history(raw, ticker)
            break
        except Exception as exc:
            if attempt == 3:
                raise RuntimeError(f"{ticker}: failed after 3 attempts; no export produced") from exc
            print(f"{ticker}: retry {attempt}/3 after {type(exc).__name__}: {exc}")
            time.sleep(2 ** attempt)
    print(f"{ticker}: {len(histories[ticker]):,} bars, "
          f"{histories[ticker].index.min().date()} to {histories[ticker].index.max().date()}")


## Align histories and inspect coverage

The overlap starts at the latest first valid bar and ends at the earliest last valid bar. Within that window the notebook uses the union of observed dates and requires valid OHLCV for every ticker. Missing observations are exported in an audit CSV. This is an observed-date alignment, not an independently verified exchange calendar: dates absent from every provider series cannot be detected here.


In [ ]:
coverage = pd.DataFrame([
    {"ticker": t, "first_date": f.dropna(subset=REQUIRED).index.min(),
     "last_date": f.dropna(subset=REQUIRED).index.max(), "downloaded_rows": len(f),
     "incomplete_rows": int(f[REQUIRED].isna().any(axis=1).sum())}
    for t, f in histories.items()
]).set_index("ticker")
if coverage[["first_date", "last_date"]].isna().any().any():
    raise ValueError("At least one ticker has no complete OHLCV observations")
overlap_start = coverage.first_date.max()
overlap_end = coverage.last_date.min()
if overlap_start > overlap_end:
    raise ValueError("The six histories have no common date range")
full_history = pd.concat(
    [histories[t].add_prefix(f"{t}_") for t in TICKERS], axis=1
).sort_index()
full_history.index.name = "date"
window = full_history.loc[overlap_start:overlap_end].copy()
missing_by_ticker = pd.DataFrame({
    t: window[[f"{t}_{c}" for c in REQUIRED]].isna().any(axis=1)
    for t in TICKERS
})
missing_by_ticker.index.name = "date"
gap_audit = missing_by_ticker.loc[missing_by_ticker.any(axis=1)]
joint = window.loc[~missing_by_ticker.any(axis=1)].copy()
if joint.empty:
    raise ValueError("No complete joint observations remain")
assert joint.index.is_unique and joint.index.is_monotonic_increasing
assert not joint[[f"{t}_{c}" for t in TICKERS for c in REQUIRED]].isna().any().any()
display(coverage)
print(f"Joint history: {joint.index.min().date()} to {joint.index.max().date()}, {len(joint):,} rows")
print(f"Excluded {len(gap_audit):,} dates inside the overlap because of missing OHLCV")
if not gap_audit.empty:
    display(gap_audit)
display(joint.head(3))
display(joint.tail(3))


## Export the complete dataset

`etf_daily_joint.csv` is the main, flat dataset: one row per shared date, ticker-prefixed fields, and ISO date strings. “Complete” requires OHLCV for all six; returns may be missing at a fund's first bar. `etf_daily_full_history.csv` preserves all downloaded histories and intentionally has blanks outside each fund's lifetime. Coverage, gap audit, and metadata document what was exported. Rerunning replaces these files; historical values may change with provider revisions.


In [ ]:
paths = {
    "joint": OUTPUT_DIR / "etf_daily_joint.csv",
    "full_history": OUTPUT_DIR / "etf_daily_full_history.csv",
    "coverage": OUTPUT_DIR / "etf_daily_coverage.csv",
    "gap_audit": OUTPUT_DIR / "etf_daily_missing_dates.csv",
    "metadata": OUTPUT_DIR / "etf_daily_metadata.json",
}
metadata = {
    "tickers": TICKERS, "source": "Yahoo Finance via yfinance",
    "downloaded_at_utc": datetime.now(ZoneInfo("UTC")).isoformat(),
    "requested_start": START, "end_exclusive": END_EXCLUSIVE,
    "joint_first_date": joint.index.min().date().isoformat(),
    "joint_last_date": joint.index.max().date().isoformat(),
    "joint_rows": len(joint), "full_history_rows": len(full_history),
    "excluded_dates_in_overlap": len(gap_audit),
    "auto_adjust": False, "repair": False, "fill_method": None,
    "return_definition": "adj_close / previous observed adj_close - 1, before alignment",
    "versions": {p: version(p) for p in ["yfinance", "pandas", "numpy"]},
    "columns": list(joint.columns),
}
for label, frame in [("joint", joint), ("full_history", full_history),
                     ("coverage", coverage), ("gap_audit", gap_audit)]:
    frame.to_csv(paths[label], date_format="%Y-%m-%d", float_format="%.12g")
paths["metadata"].write_text(json.dumps(metadata, indent=2) + "\n")
# Read back the primary deliverable to catch serialization/schema problems.
check = pd.read_csv(paths["joint"], index_col="date", parse_dates=["date"])
assert check.shape == joint.shape
assert check.index.equals(joint.index)
np.testing.assert_allclose(check.to_numpy(), joint.to_numpy(), rtol=1e-10, atol=1e-12, equal_nan=True)
for label, path in paths.items():
    print(f"{label}: {path.resolve()} ({path.stat().st_size:,} bytes)")
print("Complete joint CSV exported and read-back verified.")
